In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

In [2]:
train = pd.read_parquet('train_fe.parquet')
test = pd.read_parquet('test_fe.parquet')

print(train.shape, test.shape)
print(train['datetime'].min(), train['datetime'].max())

(84396, 34) (21780, 34)
2023-01-01 06:00:00 2025-09-18 18:00:00


In [3]:
GAP_START = pd.Timestamp('2025-02-03 18:00:00')
GAP_END   = pd.Timestamp('2025-03-01 06:00:00')

# Corrupt window per fitur rolling (locked)
ROLLING_CORRUPT_END = {
    'rolling_rain_48h': pd.Timestamp('2025-03-03 06:00:00'),
    'rolling_rain_72h': pd.Timestamp('2025-03-04 06:00:00'),
    'rolling_rain_7d':  pd.Timestamp('2025-03-08 06:00:00'),
}

In [4]:
FOLDS = [
    {'train_end': '2024-04-30', 'val_start': '2024-05-01', 'val_end': '2024-07-31'},
    {'train_end': '2024-07-31', 'val_start': '2024-08-01', 'val_end': '2024-10-31'},
    {'train_end': '2024-10-31', 'val_start': '2024-11-01', 'val_end': '2025-02-28'},
    {'train_end': '2025-01-20', 'val_start': '2025-01-21', 'val_end': '2025-09-18'},  # v2: 240 hari
]

In [5]:
FEATURES_SLOT2 = [
    # Temporal
    "bulan", "day_of_year", "hour", "days_since_last_valid_tma",
    # Identity
    "nama_pos", "latitude", "longitude",
    "tma_mean_pos", "tma_std_pos", "ac_lag1_pos",
    # Exogenous (tanpa nino_34)
    "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "dew_point_c", "cloud_cover_pct",
    "temperature_c", "wind_speed_kmh", "wind_direction_deg",
    "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_amplitude", "mjo_phase",
    # Engineered Rolling
    "rolling_rain_48h", "rolling_rain_72h", "rolling_rain_7d",
    # Horizon
    "horizon_days", "horizon_bucket",
]

CAT_FEATURES = ["nama_pos", "horizon_bucket"]
TARGET = "tma_mdpl"

assert 'nino_34' not in FEATURES_SLOT2, "nino_34 harus di-drop di Slot 2!"

In [6]:
OUTLIER_POS = ['Gunungsari', 'Wonogiri Dam']
TOP5_ERROR_POS = ['Ngadipiro', 'Ngrembang', 'Wonogiri Dam', 'Badegan', 'Karanggeneng']

In [7]:
def compute_pos_stats(train_fold):
    """Hitung tma_mean_pos, tma_std_pos, ac_lag1_pos dari train fold only."""
    stats = train_fold.groupby('nama_pos')[TARGET].agg(['mean', 'std']).reset_index()
    stats.columns = ['nama_pos', 'tma_mean_pos_new', 'tma_std_pos_new']
    # ac_lag1 dihitung terpisah per pos, sorted by datetime
    ac_list = []
    for pos, g in train_fold.sort_values('datetime').groupby('nama_pos'):
        vals = g[TARGET].values
        if len(vals) > 1:
            ac = np.corrcoef(vals[:-1], vals[1:])[0,1]
        else:
            ac = np.nan
        ac_list.append({'nama_pos': pos, 'ac_lag1_pos_new': ac})
    ac_df = pd.DataFrame(ac_list)
    return stats.merge(ac_df, on='nama_pos', how='left')    

In [8]:
def run_fold(train, fold_cfg, features, cat_features, params=None):
    train_end = pd.Timestamp(fold_cfg['train_end'])
    val_start = pd.Timestamp(fold_cfg['val_start'])
    val_end   = pd.Timestamp(fold_cfg['val_end'])

    tr = train[train['datetime'] <= train_end].copy()
    val = train[(train['datetime'] >= val_start) & (train['datetime'] <= val_end)].copy()

    # Mask corrupt rolling features near gap (jika fold menyentuh window ini)
    for feat, corrupt_end in ROLLING_CORRUPT_END.items():
        if feat in tr.columns:
            mask = (tr['datetime'] >= GAP_END) & (tr['datetime'] <= corrupt_end)
            tr.loc[mask, feat] = np.nan

    # Recompute pos-level stats dari tr only (leakage-safe)
    pos_stats = compute_pos_stats(tr)
    tr = tr.drop(columns=['tma_mean_pos','tma_std_pos','ac_lag1_pos']).merge(pos_stats, on='nama_pos', how='left')
    val = val.drop(columns=['tma_mean_pos','tma_std_pos','ac_lag1_pos']).merge(pos_stats, on='nama_pos', how='left')
    tr = tr.rename(columns=lambda c: c.replace('_new',''))
    val = val.rename(columns=lambda c: c.replace('_new',''))

    for c in cat_features:
        tr[c] = tr[c].astype('category')
        val[c] = val[c].astype('category')

    default_params = dict(
        objective='regression', metric='rmse', random_state=SEED,
        n_estimators=2000, learning_rate=0.03, num_leaves=63,
        verbosity=-1
    )
    if params: default_params.update(params)

    model = lgb.LGBMRegressor(**default_params)
    model.fit(
        tr[features], tr[TARGET],
        eval_set=[(val[features], val[TARGET])],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    preds = model.predict(val[features])
    rmse = np.sqrt(mean_squared_error(val[TARGET], preds))
    val_out = val[['datetime','nama_pos', TARGET]].copy()
    val_out['pred'] = preds
    return model, rmse, val_out

In [9]:
oof_results = []
models = []
for i, fold_cfg in enumerate(FOLDS):
    model, rmse, val_out = run_fold(train, fold_cfg, FEATURES_SLOT2, CAT_FEATURES)
    print(f"Fold {i+1}: RMSE = {rmse:.4f}")
    oof_results.append(val_out)
    models.append(model)

oof_all = pd.concat(oof_results)
oof_rmse = np.sqrt(mean_squared_error(oof_all[TARGET], oof_all['pred']))
print(f"\nOOF RMSE (Slot 2, no nino_34): {oof_rmse:.4f}")
print(f"Comparison — Slot 1 (with nino_34): 1.3052")

Fold 1: RMSE = 1.1519
Fold 2: RMSE = 1.6010
Fold 3: RMSE = 1.3303
Fold 4: RMSE = 1.3319

OOF RMSE (Slot 2, no nino_34): 1.3561
Comparison — Slot 1 (with nino_34): 1.3052


In [10]:
pos_rmse = oof_all.groupby('nama_pos').apply(
    lambda g: np.sqrt(mean_squared_error(g[TARGET], g['pred']))
).sort_values(ascending=False)
print(pos_rmse.head(10))
print("\nOutlier pos check:")
print(pos_rmse.loc[OUTLIER_POS])

nama_pos
Gunungsari                   3.620045
Karangnongko                 2.521160
Peren                        2.313137
Kali Anyar - Kreteg Abang    2.078193
Bojonegoro - Kali Kethek     2.010149
Wonogiri Dam                 1.925948
Floodway Bridge C            1.611577
Napel                        1.538443
Ketonggo                     1.317163
Bengkelolor                  1.219783
dtype: float64

Outlier pos check:
nama_pos
Gunungsari      3.620045
Wonogiri Dam    1.925948
dtype: float64


In [11]:
# Bandingkan OOF: 1 model global vs 3 model terpisah per horizon_bucket
def run_fold_bucketed(train, fold_cfg, features, cat_features):
    """Sama seperti run_fold, tapi train 1 model per horizon_bucket lalu gabung prediksi val."""
    # val di training set semua horizon_days = 0 (train tidak punya horizon).
    # Jadi test bucket split ini HANYA valid kalau val fold mengandung variasi horizon_days.
    # -> PERLU CEK DULU: apakah horizon_days sudah didefinisikan juga untuk baris train
    #    dengan cara yang sama seperti test (jarak dari observasi TMA valid terakhir)?
    pass  # isi setelah verifikasi definisi horizon_days di train

# Jika horizon_days valid di val fold, jalankan dan bandingkan:
# oof_rmse_bucketed vs oof_rmse (single model, dari sel 9)
# Keputusan: pakai bucket split HANYA jika oof_rmse_bucketed < oof_rmse secara meyakinkan (>0.01-0.02)

In [12]:
# Retrain final model pakai FULL train.csv (bukan per-fold)
pos_stats_full = compute_pos_stats(train)
train_full = train.drop(columns=['tma_mean_pos','tma_std_pos','ac_lag1_pos']).merge(pos_stats_full, on='nama_pos', how='left')
train_full = train_full.rename(columns=lambda c: c.replace('_new',''))
test_full = test.drop(columns=['tma_mean_pos','tma_std_pos','ac_lag1_pos'], errors='ignore').merge(pos_stats_full, on='nama_pos', how='left')
test_full = test_full.rename(columns=lambda c: c.replace('_new',''))

for c in CAT_FEATURES:
    train_full[c] = train_full[c].astype('category')
    test_full[c] = test_full[c].astype(pd.CategoricalDtype(categories=train_full[c].cat.categories))

final_model = lgb.LGBMRegressor(
    objective='regression', metric='rmse', random_state=SEED,
    n_estimators=int(np.mean([m.best_iteration_ for m in models])),
    learning_rate=0.03, num_leaves=63, verbosity=-1
)
final_model.fit(train_full[FEATURES_SLOT2], train_full[TARGET])
test_preds = final_model.predict(test_full[FEATURES_SLOT2])

In [13]:
assert len(test_preds) == len(test), "Jumlah prediksi tidak match test set!"
assert not np.isnan(test_preds).any(), "Ada NaN di prediksi!"
print("Pred range:", test_preds.min(), test_preds.max())
print("Train target range:", train[TARGET].min(), train[TARGET].max())
# Cek prediksi tidak absurd (misal negatif atau jauh melebihi range historis per pos)

Pred range: 1.0933089849009556 143.60418191740723
Train target range: 0.01 325.83


In [14]:
submission = pd.read_csv('sample_submission.csv')  # load dulu sample_submission.csv
submission['tma_mdpl'] = test_preds  # sesuaikan nama kolom target
submission.to_csv('submission_exp2_slot2.csv', index=False)